# Banking Customer Behaviour & Asset Analytics
### Extraction → profiling → justified cleaning → segmentation → interactive analysis

**Deliverables:** this reproducible cleaning notebook, exported datasets/audit reports, and the BankScope dashboard website.

The default source is `data/banking_raw.csv`, copied from the supplied 70-country file. Its `DataOrigin` explicitly says **Synthetic classroom simulation**. It is not live banking data. The alternative `data/bank_churn_original.csv` is supplied separately and is not merged with it.

**Important:** imputation cannot recover the true missing value. It is a documented assumption for exploratory analysis. Never impute customer IDs, outcomes, exchange rates or provenance. Preserve the raw file and report unresolved issues instead of forcing completeness to 100%.

Run all cells in order. Python dependencies: pandas, numpy, plotly, openpyxl (see `requirements.txt`). The utilities below are embedded so this notebook can travel with just its input CSV.

In [1]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 40)
ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'banking_raw.csv'
if not DATA_PATH.exists():
    candidates = [ROOT/'banking_raw.csv', ROOT/'banking_70_countries_10000_messy.csv']
    DATA_PATH = next((p for p in candidates if p.exists()), DATA_PATH)
assert DATA_PATH.exists(), 'Place the supplied CSV in data/banking_raw.csv, or change DATA_PATH.'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
print('Input:', DATA_PATH.name)
print('SHA-256:', hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| numpy:', np.__version__)

Input: banking_raw.csv
SHA-256: 90e30f95c797ca3114617e74d0ea6ad9b77fdd6bcf362d86f40fa23b2d4d4249
Python: 3.14.2 | pandas: 2.3.3 | numpy: 2.5.3


## 1. Reusable, auditable pipeline utilities
This cell embeds the same tested engine used by the website. It defines extraction, schema mapping, missing-token recognition, domain validation, cleaning, transformations, filtering, segmentation and safe exports.

**Project validation assumptions:** adult age 18–110, credit score 300–850, tenure 0–80 and no greater than age, integer-valued counts, nonnegative recorded holdings, binary fields 0/1, positive FX and Bitcoin price. These are configurable project rules, not universal banking laws. Invalid cells become missing in an analytical copy; original strings remain available for review.

Empty/whitespace strings, NA, N/A, null, NaN, ?, -, missing and not available are missing markers. `None` means no loan when it appears in loan type; it is not discarded there. Valid zero balances remain zero.

In [2]:
"""Shared, deterministic analytics pipeline for the notebook and local website."""
import io
import re
from datetime import datetime, timezone
import numpy as np
import pandas as pd

MISSING = {'', 'na', 'n/a', 'null', 'nan', '?', '-', 'missing', 'not available'}
ALIASES = {'customerid':'customer_id', 'geography':'country', 'creditscore':'credit_score',
 'numofproducts':'products_number', 'numofproduct':'products_number', 'numproducts':'products_number',
 'hascreditcard':'credit_card', 'isactivemember':'active_member', 'estimatedsalary':'estimated_salary',
 'exited':'churn', 'fxtousd':'fx_to_usd', 'bitcoinholdingsbtc':'bitcoin_btc',
 'bitcoinpriceusd':'bitcoin_price_usd', 'fixeddepositbalance':'fixed_deposit',
 'fixeddepositratepct':'fd_rate_pct', 'fixeddeposittermmonths':'fd_term_months',
 'loanbalance':'loan_balance', 'loantype':'loan_type', 'loanratepct':'loan_rate_pct',
 'stocksbalance':'stocks', 'mutualfundsbalance':'mutual_funds', 'hasbitcoin':'has_bitcoin',
 'hasfixeddeposit':'has_fixed_deposit', 'hasloan':'has_loan', 'employmentstatus':'employment_status',
 'accounttype':'account_type', 'bankingchannel':'banking_channel', 'risktolerance':'risk_tolerance',
 'snapshotdate':'snapshot_date', 'dataorigin':'data_origin'}
NUMERIC = {'age','tenure','credit_score','balance','estimated_salary','products_number','fx_to_usd',
 'bitcoin_btc','bitcoin_price_usd','fixed_deposit','fd_rate_pct','fd_term_months','loan_balance',
 'loan_rate_pct','stocks','mutual_funds','churn','credit_card','active_member','has_bitcoin','has_fixed_deposit','has_loan'}
BINARY = {'churn','credit_card','active_member','has_bitcoin','has_fixed_deposit','has_loan'}
PROTECTED = {'customer_id','churn','surname','snapshot_date','data_origin','currency','fx_to_usd','bitcoin_price_usd'}
MONEY = {'balance','estimated_salary','fixed_deposit','loan_balance','stocks','mutual_funds'}
RANGES = {'age':(18,110), 'tenure':(0,80), 'credit_score':(300,850), 'products_number':(0,30),
 'fx_to_usd':(0.00000001,1e6), 'bitcoin_price_usd':(0.000001,1e9), 'fd_rate_pct':(0,100),
 'loan_rate_pct':(0,100), 'fd_term_months':(0,1200)}
DICTIONARY = {'customer_id':'Source customer identifier; never imputed. Repeated IDs require review.',
 'country':'Country recorded by the source; no inference from names or currencies.',
 'churn':'Observed outcome: 1 = churned, 0 = retained. Missing outcomes are excluded from rate denominators.',
 'balance':'Account balance in the row currency. Zero is a valid balance.',
 'fx_to_usd':'Snapshot exchange rate: multiply local currency by this value to obtain USD.',
 'bitcoin_btc':'Bitcoin quantity in BTC; multiply by the supplied USD price, not the FX rate.',
 'loan_balance':'Liability, shown separately from assets.',
 'estimated_salary':'Estimated salary in row currency; not bank revenue.',
 'credit_score':'Source credit score; project validation assumption: 300–850.',
 'age':'Age in years; project validation assumption: adult customers aged 18–110.',
 'tenure':'Bank relationship duration in years; must not exceed age.',
 'snapshot_date':'Snapshot date, not transaction history. Charts do not imply historical trends.',
 'data_origin':'Source provenance. The 70-country file declares synthetic classroom simulation.'}

def read_csv(source):
    df = pd.read_csv(source, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    if df.empty or len(df.columns)>150 or len(df)>100000:
        raise ValueError('Use a nonempty CSV with at most 100,000 rows and 150 columns.')
    mapping = {c:ALIASES.get(re.sub(r'[^a-z0-9]', '', c.lower()), re.sub(r'\W+', '_', c.strip().lower())) for c in df}
    if len(set(mapping.values())) != len(mapping):
        raise ValueError('Multiple headers map to the same column. Rename ambiguous columns first.')
    return df.rename(columns=mapping), mapping

def blank_mask(series, column):
    s = series.astype('string').str.strip().str.lower()
    markers = MISSING | ({'none'} if column != 'loan_type' else set())
    return series.isna() | s.isin(markers)

def canonical(raw):
    """Typed analytical view. Invalid cells become NA; raw remains unchanged."""
    df = raw.copy(deep=True)
    issues = []
    for c in df:
        missing = blank_mask(df[c], c)
        s = df[c].astype('string').str.strip().mask(missing)
        invalid = pd.Series(False, index=df.index)
        if c in NUMERIC or c.endswith(('_zscore','_normalized')):
            if c in BINARY:
                s = s.str.lower().replace({'yes':'1','no':'0','true':'1','false':'0','active':'1','inactive':'0', 'churned':'1','retained':'0'})
            # Accept properly grouped thousands; never strip arbitrary punctuation.
            s = s.str.replace(r'(?<=\d),(?=\d{3}(?:\D|$))','',regex=True)
            out = pd.to_numeric(s, errors='coerce').astype(float)
            invalid = (~missing & out.isna()) | (~np.isfinite(out) & out.notna())
            if c in BINARY: invalid |= out.notna() & ~out.isin([0,1])
            if c in RANGES: invalid |= out.notna() & ~out.between(*RANGES[c])
            if c in MONEY | {'bitcoin_btc'}: invalid |= out.notna() & (out<0)
            if c in {'age','tenure','credit_score','products_number','fd_term_months'}:
                invalid |= out.notna() & (out % 1 != 0)
            df[c] = out.mask(invalid)
        else:
            if c == 'currency': s = s.str.upper()
            elif c in {'country','gender','loan_type','industry','employment_status','account_type','banking_channel','risk_tolerance'}:
                s = s.str.title().replace({'Usa':'United States','Uk':'United Kingdom','Uae':'United Arab Emirates','M':'Male','F':'Female'}) if c in {'country','gender'} else s.str.title()
            if c == 'snapshot_date':
                parsed = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d')
                invalid = s.notna() & parsed.isna()
                s = s.mask(invalid)
            df[c] = s
        issues.append({'column':c,'text_or_null_missing':int(missing.sum()),'invalid_values':int(invalid.sum())})
    if {'age','tenure'} <= set(df):
        bad = df.tenure > df.age
        df.loc[bad,'tenure'] = np.nan
        for i in issues:
            if i['column']=='tenure': i['invalid_values'] += int(bad.sum())
    return df, pd.DataFrame(issues)

def profile(raw):
    df, issues = canonical(raw)
    rows=[]
    for c in df:
        s=df[c]; num=c in NUMERIC
        r={'column':c,'type':'identifier' if c=='customer_id' else 'binary' if c in BINARY else 'numeric' if num else 'text',
           'missing':int(s.isna().sum()),'missing_pct':round(100*s.isna().mean(),2), 'unique':int(s.nunique()),
           'zeros':int(s.eq(0).sum()) if num else None,'mode':None if s.mode().empty else s.mode().iloc[0]}
        if num:
            r.update({'mean':s.mean(),'median':s.median(),'std_sample':s.std(),'variance_sample':s.var(),
                'min':s.min(),'max':s.max(),'q1':s.quantile(.25),'q3':s.quantile(.75),'skew':s.skew()})
        r.update(issues.set_index('column').loc[c].to_dict()); rows.append(r)
    missing=int(df.isna().sum().sum())
    ids=df.customer_id.dropna() if 'customer_id' in df else pd.Series(dtype=str)
    return {'rows':len(df),'columns':len(df.columns),'missing_cells':missing,
      'completeness_pct':round(100*(1-missing/max(1,df.size)),2),
      'exact_duplicates':int(raw.duplicated().sum()),'normalized_duplicates':int(df.duplicated().sum()),
      'duplicate_ids':int(ids.duplicated().sum()),'profile':rows,
      'definition':'Completeness = 100 × (1 − missing or invalid cells / all cells). It is not a correctness score.'}

def recommendation(df,c):
    s=df[c].dropna()
    if c in PROTECTED or c in BINARY:
        return 'leave','Protected identity, outcome, provenance, exchange rate or binary field: keep missing and review; do not invent facts.'
    if s.empty: return 'leave','No observed values: mean, median and mode cannot be estimated.'
    if c not in NUMERIC:
        return 'unknown','Use Unknown to preserve missingness; mode is available only when majority-category substitution is justified.'
    if c in MONEY:
        if 'currency' in df:
            return 'group_median','Money is skewed and currencies differ: use the median within each currency, never one pooled local-currency mean. All-missing groups stay missing.'
        return 'median','Median resists extreme financial values; zero is valid. Imputation changes distributions and must be disclosed.'
    if c in {'bitcoin_btc','fd_rate_pct','fd_term_months','loan_rate_pct'}:
        return 'leave','Missing holdings or contract terms are not evidence of zero ownership. Retain for review.'
    skew=s.skew()
    if c in {'age','tenure','credit_score','products_number'} or pd.isna(skew) or abs(skew)>.5:
        return 'median','Median is robust to skew/outliers. For discrete fields the result is rounded to a whole value.'
    return 'mean','Mean uses all observed values and is suitable for an approximately symmetric numeric distribution. Check outliers first.'

def clean(raw, plan=None, deduplicate=True):
    df, issues=canonical(raw)
    audit=[{'timestamp':datetime.now(timezone.utc).isoformat(),'column':r['column'],'method':'normalize / validate',
       'affected':int(r['text_or_null_missing']+r['invalid_values']), 'reason':f"Text/null missing: {r['text_or_null_missing']}; invalid: {r['invalid_values']}. Invalid values quarantined as missing; raw preserved."} for r in issues.to_dict('records') if r['text_or_null_missing'] or r['invalid_values']]
    if deduplicate:
        # Remove only exact raw duplicates; never merge IDs or normalized lookalikes.
        n=int(raw.duplicated().sum()); df=df.loc[~raw.duplicated()].copy()
        audit.append({'column':'all','method':'exact duplicate removal','affected':n,'reason':'Identical source rows only; conflicting IDs retained.'})
    plan = plan if plan is not None else {c:recommendation(df,c)[0] for c in df}
    recommendations={c:recommendation(df,c) for c in df}
    for c,config in plan.items():
        if c not in df: raise ValueError(f'Unknown column: {c}')
        cfg=config if isinstance(config,dict) else {'method':config}
        method=cfg.get('method','leave'); missing=df[c].isna(); n=int(missing.sum())
        if method=='leave' or n==0: continue
        if c in PROTECTED and method!='drop': raise ValueError(f'{c} is protected. Keep missing or explicitly remove incomplete rows.')
        if method=='drop': df=df.loc[~missing].copy(); value='row removed'
        elif method in {'mean','median','group_median'}:
            if c not in NUMERIC or c in BINARY: raise ValueError('Numeric imputation requires a continuous numeric field.')
            if method=='group_median':
                if 'currency' not in df: raise ValueError('Currency column required for currency-group imputation.')
                value=df.groupby('currency',dropna=True)[c].transform('median')
            else: value=getattr(df[c],method)()
            if c in {'age','tenure','credit_score','products_number'}: value=np.round(value)
            df[c]=df[c].fillna(value)
        elif method=='mode':
            mode=df[c].mode()
            value=mode.iloc[0] if len(mode) else np.nan
            df[c]=df[c].fillna(value)
        elif method in {'unknown','constant'}:
            value='Unknown' if method=='unknown' else cfg.get('value')
            if c in NUMERIC:
                try: value=float(value)
                except (ValueError,TypeError): raise ValueError('Provide a numeric replacement.')
            df[c]=df[c].fillna(value)
        else: raise ValueError(f'Unsupported cleaning method: {method}')
        if method!='drop':
            validated,_=canonical(df)
            if (missing & df[c].notna() & validated[c].isna()).any(): raise ValueError('Replacement violates the column validation rules.')
            if c in {'age','tenure'} and {'age','tenure'}<=set(df) and (df.tenure>df.age).any():
                raise ValueError('Imputation creates tenure greater than age. Choose another method or retain missing values.')
        affected=n if method=='drop' else n-int(df[c].isna().sum())
        audit.append({'column':c,'method':method,'affected':affected,'reason':recommendations[c][1] if config==recommendations[c][0] else 'Explicit user-selected method; review assumptions before applying.',
            'replacement':'within-currency median' if method=='group_median' else str(value)})
    stamp=datetime.now(timezone.utc).isoformat()
    for a in audit: a.setdefault('timestamp',stamp)
    return df.reset_index(drop=True),audit

def transform(frame):
    df=frame.copy()
    bands={'age':([0,25,35,45,60,110],['18–25','26–35','36–45','46–60','61+'],'age_band'),
       'credit_score':([299,579,669,739,799,850],['300–579','580–669','670–739','740–799','800–850'],'credit_band'),
       'tenure':([-1,2,5,8,80],['0–2','3–5','6–8','9+'],'tenure_band')}
    for c,(bins,labels,name) in bands.items():
        if c in df: df[name]=pd.cut(df[c],bins,labels=labels).astype('string')
    for c,name,labels in [('churn','churn_label',{0:'Retained',1:'Churned'}),('active_member','activity_label',{0:'Inactive',1:'Active'}),('credit_card','card_label',{0:'No card',1:'Has card'})]:
        if c in df: df[name]=df[c].map(labels).astype('string').fillna('Unknown')
    if 'fx_to_usd' in df:
        for c in MONEY & set(df): df[c+'_usd']=df[c]*df.fx_to_usd
    if {'bitcoin_btc','bitcoin_price_usd'} <= set(df): df['bitcoin_usd']=df.bitcoin_btc*df.bitcoin_price_usd
    if 'balance_usd' in df:
        df['balance_band']=pd.cut(df.balance_usd,[-.01,0,10000,50000,100000,np.inf],labels=['Zero','0–10k','10–50k','50–100k','100k+']).astype('string')
    return df

def filter_frame(df,query=None):
    q=query or {}; masks=[]
    for c,values in q.get('categories',{}).items():
        if c in df and values:
            if pd.api.types.is_numeric_dtype(df[c]):
                masks.append(df[c].isin(pd.to_numeric(pd.Series(values),errors='coerce').dropna()))
            else: masks.append(df[c].fillna('Unknown').astype(str).isin([str(v) for v in values]))
    for c,bounds in q.get('ranges',{}).items():
        if c in df and bounds and len(bounds)==2:
            masks.append(pd.to_numeric(df[c],errors='coerce').between(float(bounds[0]),float(bounds[1])))
    if masks:
        mask=masks[0]
        for m in masks[1:]: mask=mask|m if q.get('logic')=='OR' else mask&m
        df=df.loc[mask]
    term=str(q.get('search','')).strip()
    if term: df=df.loc[df.astype('string').apply(lambda x:x.str.contains(term,case=False,regex=False,na=False)).any(axis=1)]
    sort=q.get('sort',[])
    if sort:
        cols=[s['column'] for s in sort if s['column'] in df]
        if cols: df=df.sort_values(cols,ascending=[s.get('ascending',True) for s in sort if s['column'] in df],kind='stable',na_position='last')
    return df

def segments(df,by):
    if by not in df: return pd.DataFrame()
    groups=df.groupby(by,dropna=False,observed=True)
    out=groups.size().rename('customers').to_frame()
    out['share_pct']=100*out.customers/max(1,len(df))
    if 'churn' in df:
        out['known_outcomes']=groups.churn.count(); out['churned']=groups.churn.sum(min_count=1)
        out['retained']=out.known_outcomes-out.churned
        out['churn_rate_pct']=100*out.churned/out.known_outcomes.replace(0,np.nan)
    for c in ['age','credit_score','balance_usd','estimated_salary_usd','bitcoin_usd','fixed_deposit_usd','stocks_usd','mutual_funds_usd','loan_balance_usd']:
        if c in df:
            out['mean_'+c]=groups[c].mean(); out['observed_'+c]=groups[c].count()
    out['small_sample']=out.customers<30
    return out.reset_index().sort_values('customers',ascending=False)

def outliers(df,method='IQR'):
    rows=[]
    for c in df.select_dtypes('number'):
        if c in BINARY or c=='customer_id': continue
        # Original money must be inspected within currency; compare normalized USD here.
        if c in MONEY and 'currency' in df: continue
        s=df[c]; q1=s.quantile(.25);q3=s.quantile(.75);iqr=q3-q1
        lo,hi=(q1-1.5*iqr,q3+1.5*iqr) if method=='IQR' else (s.mean()-3*s.std(),s.mean()+3*s.std())
        rows.append({'column':c,'q1':q1,'q3':q3,'iqr':iqr,'lower':lo,'upper':hi,'flagged':int(((s<lo)|(s>hi)).sum()),'method':method})
    return pd.DataFrame(rows)

def statistics(df):
    n=df.select_dtypes('number').drop(columns=['customer_id'],errors='ignore')
    if 'currency' in df: n=n.drop(columns=list(MONEY),errors='ignore')
    if n.empty: return pd.DataFrame()
    result=n.describe(percentiles=[.05,.25,.5,.75,.95]).T
    result['median']=n.median();result['variance_sample']=n.var(); result['skew']=n.skew()
    result['iqr']=n.quantile(.75)-n.quantile(.25)
    result['range']=n.max()-n.min()
    result['mode']=n.apply(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
    return result.reset_index(names='column')

def records(df):
    import json
    return json.loads(df.to_json(orient='records',date_format='iso'))

def excel_safe(df):
    """Prevent exported text from being interpreted as spreadsheet formulas."""
    d=df.copy()
    for c in d.select_dtypes(include=['object','string']).columns:
        d[c]=d[c].map(lambda x:"'"+x if isinstance(x,str) and x.lstrip().startswith(('=','+','-','@','\t','\r')) else x)
    return d

def workbook(df,audit,quality):
    b=io.BytesIO()
    with pd.ExcelWriter(b,engine='openpyxl') as w:
        excel_safe(df).to_excel(w,sheet_name='Active data',index=False)
        statistics(df).to_excel(w,sheet_name='Statistics',index=False)
        excel_safe(pd.DataFrame(audit)).to_excel(w,sheet_name='Cleaning audit',index=False)
        pd.DataFrame(quality['profile']).to_excel(w,sheet_name='Quality',index=False)
        if 'country' in df: excel_safe(segments(df,'country')).to_excel(w,sheet_name='Countries',index=False)
        if {'country','churn_label'}<=set(df):
            excel_safe(pd.crosstab(df.country.fillna('Unknown'),df.churn_label.fillna('Unknown')).reset_index()).to_excel(w,sheet_name='Country outcome cross-tab',index=False)
        if {'country','products_number'}<=set(df):
            excel_safe(df.groupby(['country','products_number'],dropna=False).size().reset_index(name='records')).to_excel(w,sheet_name='Product distribution',index=False)
    return b.getvalue()


## 2. Extract and inspect the actual schema
Read as strings first so pandas does not silently interpret legitimate category labels as missing. Header mapping is explicit and collisions are rejected.

In [3]:
raw, mapping = read_csv(DATA_PATH)
raw_snapshot = raw.copy(deep=True)
display(pd.DataFrame(mapping.items(), columns=['source_header','analytical_header']))
print('Shape:', raw.shape)
display(raw.head(10))
display(raw.tail(10))
display(pd.DataFrame([{'column':c,'definition':DICTIONARY.get(c,c.replace('_',' '))} for c in raw]))
print('Provenance:', raw.get('data_origin',pd.Series(dtype=str)).unique())

,source_header,analytical_header
0,CustomerId,customer_id
1,Geography,country
2,Gender,gender
3,Age,age
4,Tenure,tenure
5,CreditScore,credit_score
6,Currency,currency
7,FXToUSD,fx_to_usd
8,Balance,balance
9,EstimatedSalary,estimated_salary


Shape: (10000, 34)


,customer_id,country,gender,age,tenure,credit_score,currency,fx_to_usd,balance,estimated_salary,bitcoin_btc,bitcoin_price_usd,fixed_deposit,fd_rate_pct,fd_term_months,loan_balance,loan_type,loan_rate_pct,stocks,mutual_funds,has_bitcoin,has_fixed_deposit,has_loan,products_number,credit_card,active_member,churn,industry,employment_status,account_type,banking_channel,risk_tolerance,snapshot_date,data_origin
0,SYN004992,China,Female,82,38,593,CNY,0.14,47004.57,124397.93,0.01454737,60000,1057795.5,1.81,24,0.0,None,0,398589.5,68225.86,1,1,0,5,0,1,0,Manufacturing,Unemployed,Savings,Branch,High,2026-09-30,Synthetic classroom simulation
1,SYN006753,Japan,Female,23,0,695,JPY,0.0067,3682682.09,5648737.31,0.12915452,60000,0.0,0,0,10200849.25,Business,9.01,10887317.91,454935.82,1,0,1,5,0,1,0,Construction,Employed,Savings,Online,High,2026-09-30,Synthetic classroom simulation
2,SYN003648,Denmark,Male,abc,29,756,DKK,0.15,60293.2,974035.13,0,60000,NA,0,0,0.0,None,0,1207645.27,0.0,0,0,0,3,1,0,0,,Retired,savings,Mobile,Medium,2026-09-30,Synthetic classroom simulation
3,SYN007754,Indonesia,Female,82,18,665,IDR,6.3e-05,2462060317.46,716086666.67,0,60000,96832063.49,4.59,60,1382740952.38,Home,8.27,0.0,864683174.6,0,1,Yes,4,0,0,1,Education,Retired,Savings,Branch,Medium,2026-09-30,Synthetic classroom simulation
4,SYN004016,United States,Male,81,24,682,USD,1,32057.94,15748.51,0,60000,0.0,0,0,0.0,None,0,11690.41,0.0,0,0,0,3,1,0,0,Manufacturing,Student,Premium,Online,Low,2026-09-30,Synthetic classroom simulation
5,SYN009459,United Kingdom,Male,25,3,568,GBP,1.25,0.0,9996.97,0,60000,16106.82,7.6,24,0.0,None,0,3452.9,20733.54,0,1,0,5,1,1,0,Manufacturing,Unemployed,Current,Mobile,High,2026-09-30,Synthetic classroom simulation
6,SYN008883,Uzbekistan,Male,19,1,648,UZS,8e-05,1043157125.0,1314766125.0,0.06728208,60000,0.0,0,0,71763125.0,Education,12.62,0.0,0.0,1,0,1,3,0,1,0,Construction,Self-employed,Salary,Branch,Low,2026-09-30,Synthetic classroom simulation
7,SYN004699,Sweden,Female,41,4,709,SEK,0.095,0.0,186279.16,0,60000,105602.0,6.2,12,0.0,None,0,0.0,0.0,0,1,0,3,1,1,0,Construction,Employed,Salary,Online,Low,2026-09-30,Synthetic classroom simulation
8,SYN005919,Ecuador,Male,64,11,650,USD,1,missing,45364.83,0.205904,60000,25049.4,7.0,24,12232.62,Personal,6.89,0.0,0.0,1,1,1,5,1,0,0,Technology,Retired,Premium,Online,Low,2026-09-30,Synthetic classroom simulation
9,SYN000264,Singapore,Female,66,29,814,SGD,0.74,4323.15,74095.69,0,60000,14586.47,1.44,60,71437.78,Business,3.63,0.0,17761.66,0,1,1,4,0,1,1,Construction,Student,Salary,Branch,High,2026-09-30,Synthetic classroom simulation


,customer_id,country,gender,age,tenure,credit_score,currency,fx_to_usd,balance,estimated_salary,bitcoin_btc,bitcoin_price_usd,fixed_deposit,fd_rate_pct,fd_term_months,loan_balance,loan_type,loan_rate_pct,stocks,mutual_funds,has_bitcoin,has_fixed_deposit,has_loan,products_number,credit_card,active_member,churn,industry,employment_status,account_type,banking_channel,risk_tolerance,snapshot_date,data_origin
9990,SYN006866,South Africa,Female,27,4,535,ZAR,0.055,445622.36,202196.73,0,60000,0.0,0,0,148589.64,Auto,5.87,616969.27,635174.0,0,0,1,4,0,0,0,Hospitality,Employed,Premium,Online,High,2026-09-30,Synthetic classroom simulation
9991,SYN001750,Ecuador,Male,50,7,542,USD,1,4312.04,11663.71,0,60000,0.0,0,0,107710.93,Home,12.98,3624.43,0.0,0,0,1,3,0,0,1,Technology,Employed,Savings,Mobile,High,2026-09-30,Synthetic classroom simulation
9992,SYN007218,Uganda,Female,60,8,774,UGX,0.00027,67897185.19,54311592.59,0,60000,0.0,0,0,0.0,None,0,44473777.78,0.0,0,0,0,2,0,1,0,Manufacturing,Retired,Premium,Mixed,Medium,2026-09-30,Synthetic classroom simulation
9993,SYN006358,Rwanda,Female,58,1,652,RWF,0.00075,23421520.0,28130080.0,0,60000,0.0,0,0,8350026.67,Personal,5.53,0.0,NaN,0,0,1,3,0,1,0,Public Services,Unemployed,Salary,Online,Low,2026-09-30,Synthetic classroom simulation
9994,SYN009089,Saudi Arabia,Male,46,3,833,SAR,0.267,0.0,61565.62,0,60000,0.0,0,0,0.0,None,0,0.0,0.0,0,0,0,2,1,1,0,Transport,Student,Salary,Mixed,Low,2026-09-30,Synthetic classroom simulation
9995,SYN000647,Italy,Male,63,23,615,EUR,1.1,0.0,69302.42,0,60000,21154.07,4.94,60,0.0,None,0,0.0,22704.4,0,1,0,4,1,1,0,Manufacturing,Unemployed,Current,Mobile,Medium,2026-09-30,Synthetic classroom simulation
9996,SYN004851,Malaysia,Male,44,24,714,MYR,0.22,22929.09,106634.27,0,60000,0.0,0,0,476731.91,Business,7.83,0.0,0.0,0,0,1,3,1,0,0,Technology,Self-employed,Current,Branch,High,2026-09-30,Synthetic classroom simulation
9997,SYN008520,Germany,Female,54,7,780,EUR,1.1,6868.18,5637.36,0,60000,5121.94,3.52,60,0.0,None,0,15695.4,0.0,0,1,0,4,1,1,0,Hospitality,Employed,Current,Branch,Medium,2026-09-30,Synthetic classroom simulation
9998,SYN008501,Thailand,Female,62,23,605,THB,0.028,882563.93,1800544.29,0,60000,5173161.43,1.9,36,0.0,None,0,0.0,0.0,0,1,0,2,0,1,0,Other,Retired,Current,Branch,High,2026-09-30,Synthetic classroom simulation
9999,SYN006655,Indonesia,Female,72,2,488,IDR,6.3e-05,126914285.71,341382063.49,0,60000,545971904.76,2.36,24,0.0,None,0,35497936.51,0.0,0,1,0,3,0,1,0,Manufacturing,Retired,Premium,Mixed,Low,2026-09-30,Synthetic classroom simulation


,column,definition
0,customer_id,Source customer identifier; never imputed. Rep...
1,country,Country recorded by the source; no inference f...
2,gender,gender
3,age,Age in years; project validation assumption: a...
4,tenure,Bank relationship duration in years; must not ...
5,credit_score,Source credit score; project validation assump...
6,currency,currency
7,fx_to_usd,Snapshot exchange rate: multiply local currenc...
8,balance,Account balance in the row currency. Zero is a...
9,estimated_salary,Estimated salary in row currency; not bank rev...


Provenance: ['Synthetic classroom simulation']


## 3. Profile before choosing a cleaning method
Count textual missingness separately from invalid numeric values. Exact duplicate rows and repeated customer IDs represent different problems. Never automatically merge conflicting IDs.

In [4]:
typed, conversion_issues = canonical(raw)
before = profile(raw)
display(pd.DataFrame(before['profile']))
display(conversion_issues)
print({k:v for k,v in before.items() if k not in ['profile','definition']})
print(before['definition'])
duplicate_ids = typed.loc[typed.customer_id.notna() & typed.customer_id.duplicated(keep=False)]
display(duplicate_ids.head(20))
invalid_examples=[]
for c in raw:
    mask = ~blank_mask(raw[c],c) & typed[c].isna()
    for row in raw.index[mask][:8]:
        invalid_examples.append({'source_row':int(row)+2,'column':c,'original_value':raw.loc[row,c]})
display(pd.DataFrame(invalid_examples))

,column,type,missing,missing_pct,unique,zeros,mode,text_or_null_missing,invalid_values,mean,median,std_sample,variance_sample,min,max,q1,q3,skew
0,customer_id,identifier,0,0.00,9700,NaN,SYN001650,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,country,text,0,0.00,70,NaN,China,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gender,text,128,1.28,4,NaN,Female,128,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,age,numeric,211,2.11,68,0.0,54.0,152,59,5.154367e+01,52.00,1.957648e+01,3.832387e+02,18.00000,8.500000e+01,35.0000,6.900000e+01,0.006687
4,tenure,numeric,228,2.28,41,723.0,0.0,155,73,1.395589e+01,11.00,1.138883e+01,1.297055e+02,0.00000,4.000000e+01,4.0000,2.200000e+01,0.605340
5,credit_score,numeric,219,2.19,435,0.0,850.0,133,86,6.589957e+02,660.00,8.331858e+01,6.941985e+03,362.00000,8.500000e+02,602.0000,7.160000e+02,-0.071665
6,currency,text,128,1.28,58,NaN,EUR,128,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,fx_to_usd,numeric,0,0.00,52,0.0,1.1,0,0,4.793693e-01,0.14,6.739209e-01,4.541694e-01,0.00004,3.250000e+00,0.0075,1.100000e+00,1.964157
8,balance,numeric,159,1.59,7986,1675.0,0.0,129,30,1.052072e+08,69213.88,4.688938e+09,2.198614e+19,0.00000,4.083288e+11,6426.0800,1.300022e+06,77.451983
9,estimated_salary,numeric,161,1.61,9546,0.0,15047.73,130,31,6.534206e+07,285084.93,8.161474e+08,6.660966e+17,1710.35000,5.777494e+10,47191.5500,4.909240e+06,51.160160


,column,text_or_null_missing,invalid_values
0,customer_id,0,0
1,country,0,0
2,gender,128,0
3,age,152,59
4,tenure,155,73
5,credit_score,133,86
6,currency,128,0
7,fx_to_usd,0,0
8,balance,129,30
9,estimated_salary,130,31


{'rows': 10000, 'columns': 34, 'missing_cells': 2527, 'completeness_pct': 99.26, 'exact_duplicates': 200, 'normalized_duplicates': 200, 'duplicate_ids': 300}
Completeness = 100 × (1 − missing or invalid cells / all cells). It is not a correctness score.


,customer_id,country,gender,age,tenure,credit_score,currency,fx_to_usd,balance,estimated_salary,bitcoin_btc,bitcoin_price_usd,fixed_deposit,fd_rate_pct,fd_term_months,loan_balance,loan_type,loan_rate_pct,stocks,mutual_funds,has_bitcoin,has_fixed_deposit,has_loan,products_number,credit_card,active_member,churn,industry,employment_status,account_type,banking_channel,risk_tolerance,snapshot_date,data_origin
6,SYN008883,Uzbekistan,Male,19.0,1.0,648.0,UZS,0.00008,1.043157e+09,1.314766e+09,0.067282,60000.0,0.000000e+00,0.00,0.0,7.176312e+07,Education,12.62,0.00,0.000000e+00,1.0,0.0,1.0,3.0,0.0,1.0,0.0,Construction,Self-Employed,Salary,Branch,Low,2026-09-30,Synthetic classroom simulation
20,SYN006273,Tunisia,Male,31.0,9.0,604.0,TND,0.32000,1.233082e+05,6.517309e+04,0.000000,60000.0,0.000000e+00,0.00,0.0,2.885075e+04,Business,7.20,90414.62,2.131209e+04,0.0,0.0,1.0,4.0,0.0,0.0,1.0,Public Services,Student,Savings,Mixed,Low,2026-09-30,Synthetic classroom simulation
21,SYN003866,Uganda,Male,73.0,4.0,755.0,UGX,0.00027,0.000000e+00,6.752816e+08,0.000000,60000.0,0.000000e+00,0.00,0.0,9.307315e+07,Business,4.74,NaN,0.000000e+00,0.0,0.0,1.0,3.0,1.0,0.0,1.0,Retail,Unemployed,Savings,Mobile,Low,2026-09-30,Synthetic classroom simulation
35,SYN003672,Uruguay,Male,75.0,9.0,646.0,UYU,0.02500,7.541052e+05,7.133772e+05,0.000000,60000.0,8.801076e+05,3.47,12.0,2.537172e+06,Home,4.12,2319170.80,3.989648e+05,0.0,1.0,1.0,5.0,0.0,0.0,1.0,Education,Unemployed,Current,Mobile,High,2026-09-30,Synthetic classroom simulation
39,SYN007616,New Zealand,Male,59.0,34.0,578.0,NZD,0.61000,7.907008e+04,6.828130e+04,0.112153,60000.0,7.594977e+05,7.26,24.0,2.490252e+04,Home,4.66,17362.69,1.297587e+04,1.0,1.0,1.0,6.0,0.0,1.0,0.0,Construction,Self-Employed,Salary,Mobile,Medium,2026-09-30,Synthetic classroom simulation
51,SYN003458,Egypt,Female,58.0,3.0,687.0,EGP,0.02000,6.599960e+05,3.405085e+05,0.000000,60000.0,1.049775e+06,3.30,60.0,6.584475e+05,Education,17.42,0.00,0.000000e+00,0.0,1.0,1.0,4.0,1.0,1.0,0.0,Finance,Self-Employed,Current,Online,High,2026-09-30,Synthetic classroom simulation
61,SYN004838,Finland,Non-Binary,46.0,23.0,NaN,EUR,1.10000,1.469309e+04,3.654816e+04,0.000000,60000.0,4.385944e+04,3.02,6.0,0.000000e+00,None,0.00,0.00,0.000000e+00,0.0,1.0,0.0,2.0,0.0,1.0,0.0,Finance,Self-Employed,Savings,Review Conflict,Medium,2026-09-30,Synthetic classroom simulation
77,SYN005832,Romania,Male,69.0,28.0,786.0,RON,0.22000,5.701823e+04,NaN,0.000000,60000.0,4.661716e+05,5.42,36.0,4.762259e+04,Education,6.43,0.00,0.000000e+00,0.0,1.0,1.0,3.0,0.0,1.0,0.0,Technology,Self-Employed,Premium,Mixed,High,2026-09-30,Synthetic classroom simulation
98,SYN005088,Norway,Female,48.0,9.0,693.0,NOK,0.09500,7.701295e+04,7.735901e+05,0.000000,60000.0,0.000000e+00,0.00,0.0,4.695493e+05,Home,16.95,319698.21,0.000000e+00,0.0,0.0,1.0,4.0,1.0,0.0,0.0,Manufacturing,Retired,Premium,Branch,High,2026-09-30,Synthetic classroom simulation
110,SYN004447,Tanzania,Female,64.0,20.0,708.0,TZS,0.00038,4.644984e+07,1.841026e+08,0.000000,60000.0,2.487803e+07,2.33,24.0,1.475145e+08,Education,3.70,0.00,0.000000e+00,0.0,1.0,1.0,4.0,1.0,0.0,1.0,Hospitality,Self-Employed,Premium,Branch,High,2026-09-30,Synthetic classroom simulation


,source_row,column,original_value
0,4,age,abc
1,59,age,150
2,427,age,-5
3,697,age,220
4,711,age,0
...,...,...,...
123,2734,churn,N
124,2860,churn,N
125,3225,churn,N
126,3918,churn,N


In [5]:
quality = pd.DataFrame(before['profile'])
px.bar(quality,x='column',y='missing_pct',title='Missing / invalid cells before cleaning (%)',labels={'missing_pct':'Missing or invalid (%)'}).show()
px.imshow(typed.isna().head(300).astype(int).T,aspect='auto',color_continuous_scale=['#e9edf5','#ee7298'],title='Missingness matrix — first 300 source rows').show()

## 4. Why mean, median or mode? A defensible decision

- **Mean**: uses every observed value; useful for approximately symmetric continuous measurements without influential extremes. It is pulled toward large outliers. Do not average identifiers, binary labels or mixed local currencies.
- **Median**: the middle observed value; more robust to skew and extreme balances. For money, calculate it **within currency**. Rounded medians preserve integer counts. It reduces variance and is still only an assumption.
- **Mode**: most frequent observed value; applies to categories or discrete values. It can inflate the largest group and conceal missingness. Ties here use pandas' first sorted mode and are not evidence of a uniquely correct replacement.
- **Unknown / keep missing**: preserves uncertainty. Default text treatment is `Unknown`. IDs, churn, binary ownership/status, currency, FX and pricing fields remain missing for review. Missing Bitcoin quantities and contract terms remain missing.
- **Remove rows**: explicit option when essential fields are absent; disclose sample loss and potential selection bias. Never silently remove valid extremes.

The plan below is computed from the observed columns and their distributions. You can edit the plan before the next cell. Exploratory imputation does not justify using the same procedure for a predictive model: fit imputers on training data only to avoid leakage.

In [6]:
decision_rows=[]
for c in typed:
    method, reason = recommendation(typed,c)
    s=typed[c]
    decision_rows.append({'column':c,'missing':int(s.isna().sum()),'method':method,'why':reason,
        'skew':s.skew() if c in NUMERIC else None})
decisions=pd.DataFrame(decision_rows)
display(decisions)
plan = dict(zip(decisions.column, decisions.method))
# Examples of explicit overrides:
# plan['age'] = 'median'
# plan['gender'] = 'mode'  # Only if you can justify majority-category substitution.
# plan['industry'] = {'method':'constant','value':'Unknown'}
# plan['churn'] = 'drop'  # Explicitly drop unknown outcomes; never invent labels.
display(pd.DataFrame({'illustrative_values':[10,11,12,13,1000]}).agg(['mean','median']))
print('The small example above is illustrative, not banking records.')

,column,missing,method,why,skew
0,customer_id,0,leave,"Protected identity, outcome, provenance, excha...",NaN
1,country,0,unknown,Use Unknown to preserve missingness; mode is a...,NaN
2,gender,128,unknown,Use Unknown to preserve missingness; mode is a...,NaN
3,age,211,median,Median is robust to skew/outliers. For discret...,0.006687
4,tenure,228,median,Median is robust to skew/outliers. For discret...,0.605340
5,credit_score,219,median,Median is robust to skew/outliers. For discret...,-0.071665
6,currency,128,leave,"Protected identity, outcome, provenance, excha...",NaN
7,fx_to_usd,0,leave,"Protected identity, outcome, provenance, excha...",1.964157
8,balance,159,group_median,Money is skewed and currencies differ: use the...,77.451983
9,estimated_salary,161,group_median,Money is skewed and currencies differ: use the...,51.160160


,illustrative_values
mean,209.2
median,12.0


The small example above is illustrative, not banking records.


## 5. Apply the reviewed plan to a copy
Normalize labels, quarantine invalid entries as missing, remove exact source duplicates, then apply the documented per-column strategy. Conflicting IDs remain flagged. No default outlier removal.

In [7]:
cleaned, audit = clean(raw, plan=plan, deduplicate=True)
after = profile(cleaned)
comparison=pd.DataFrame([
    {'stage':'Before','rows':before['rows'],'missing_or_invalid_cells':before['missing_cells'],'exact_duplicates':before['exact_duplicates'],'extra_repeated_ids':before['duplicate_ids']},
    {'stage':'After','rows':after['rows'],'missing_or_invalid_cells':after['missing_cells'],'exact_duplicates':after['exact_duplicates'],'extra_repeated_ids':after['duplicate_ids']}])
display(comparison)
display(pd.DataFrame(audit))
display(pd.DataFrame(after['profile']).query('missing > 0')[['column','missing','missing_pct']])
print('Unresolved missing values are intentional when a defensible replacement is unavailable.')
pd.testing.assert_frame_equal(raw,raw_snapshot)
assert len(cleaned)==len(raw)-int(raw.duplicated().sum())
assert cleaned.churn.dropna().isin([0,1]).all()

,stage,rows,missing_or_invalid_cells,exact_duplicates,extra_repeated_ids
0,Before,10000,2527,200,300
1,After,9800,430,0,100


,timestamp,column,method,affected,reason,replacement
0,2026-10-12T04:22:24.448258+00:00,gender,normalize / validate,128,Text/null missing: 128; invalid: 0. Invalid va...,NaN
1,2026-10-12T04:22:24.448268+00:00,age,normalize / validate,211,Text/null missing: 152; invalid: 59. Invalid v...,NaN
2,2026-10-12T04:22:24.448271+00:00,tenure,normalize / validate,228,Text/null missing: 155; invalid: 73. Invalid v...,NaN
3,2026-10-12T04:22:24.448273+00:00,credit_score,normalize / validate,219,Text/null missing: 133; invalid: 86. Invalid v...,NaN
4,2026-10-12T04:22:24.448275+00:00,currency,normalize / validate,128,Text/null missing: 128; invalid: 0. Invalid va...,NaN
5,2026-10-12T04:22:24.448281+00:00,balance,normalize / validate,159,Text/null missing: 129; invalid: 30. Invalid v...,NaN
6,2026-10-12T04:22:24.448283+00:00,estimated_salary,normalize / validate,161,Text/null missing: 130; invalid: 31. Invalid v...,NaN
7,2026-10-12T04:22:24.448284+00:00,bitcoin_btc,normalize / validate,176,Text/null missing: 130; invalid: 46. Invalid v...,NaN
8,2026-10-12T04:22:24.448286+00:00,fixed_deposit,normalize / validate,171,Text/null missing: 147; invalid: 24. Invalid v...,NaN
9,2026-10-12T04:22:24.448287+00:00,loan_balance,normalize / validate,165,Text/null missing: 138; invalid: 27. Invalid v...,NaN


,column,missing,missing_pct
6,currency,125,1.28
8,balance,3,0.03
10,bitcoin_btc,173,1.77
12,fixed_deposit,2,0.02
15,loan_balance,2,0.02
18,stocks,3,0.03
20,has_bitcoin,18,0.18
21,has_fixed_deposit,20,0.20
22,has_loan,22,0.22
24,credit_card,20,0.20


Unresolved missing values are intentional when a defensible replacement is unavailable.


## 6. Missing text, currency and binary cross-checks
`Unknown` is an explicit category and must not be mistaken for observed demographic information. Remaining missing outcomes are excluded from churn denominators. Missing currency is kept for review even when FX is recorded; USD conversion uses the supplied FX only.

The audit reports affected cell counts. The cell-level provenance export records raw values, typed values, final values and whether a value was replaced. Removed exact duplicates have no final row.

In [8]:
kept = raw.loc[~raw.duplicated()].copy()
typed_kept = canonical(kept)[0].reset_index(drop=True)
provenance=[]
for c in raw:
    original = kept[c].reset_index(drop=True)
    old = typed_kept[c]
    new = cleaned[c]
    changed = old.astype('string').fillna('<MISSING>').ne(new.astype('string').fillna('<MISSING>'))
    review = changed | old.isna()
    for i in np.flatnonzero(review.to_numpy()):
        provenance.append({'source_csv_row':int(kept.index[i])+2,'customer_id':str(original.iloc[i]) if c=='customer_id' else str(cleaned.customer_id.iloc[i]),
            'column':c,'raw_value':original.iloc[i],'typed_value':old.iloc[i],'cleaned_value':new.iloc[i],
            'imputed':bool(old.isna().iloc[i] and pd.notna(new.iloc[i]))})
cell_audit=pd.DataFrame(provenance)
display(cell_audit.head(30))
print('Missing churn excluded:',int(cleaned.churn.isna().sum()))
print('Valid zero balances retained:',int(cleaned.balance.eq(0).sum()))

,source_csv_row,customer_id,column,raw_value,typed_value,cleaned_value,imputed
0,193,SYN006400,gender,N/A,<NA>,Unknown,True
1,311,SYN009537,gender,,<NA>,Unknown,True
2,314,SYN003984,gender,NaN,<NA>,Unknown,True
3,540,SYN002802,gender,NaN,<NA>,Unknown,True
4,691,SYN009005,gender,missing,<NA>,Unknown,True
5,822,SYN006146,gender,missing,<NA>,Unknown,True
6,867,SYN006084,gender,null,<NA>,Unknown,True
7,883,SYN005844,gender,,<NA>,Unknown,True
8,1111,SYN008417,gender,N/A,<NA>,Unknown,True
9,1152,SYN008233,gender,missing,<NA>,Unknown,True


Missing churn excluded: 19
Valid zero balances retained: 1636


## 7. Transform without destroying the original scale
Derive age, tenure and credit bands. Convert local money with the row-level FX rate. Bitcoin uses BTC × USD price; it is never multiplied by the local FX rate. Loans are liabilities, excluded from assets. No invented transactions, historical returns, or current market prices.

In [9]:
data=transform(cleaned)
display(data.head(10))
print('Derived columns:',[c for c in data if c not in cleaned])
display(outliers(data,'IQR'))
display(outliers(data,'Z-score'))
print('Outliers are retained. IQR flags can include legitimate zeros or large holdings.')
numeric=statistics(data)
display(numeric)

,customer_id,country,gender,age,tenure,credit_score,currency,fx_to_usd,balance,estimated_salary,bitcoin_btc,bitcoin_price_usd,fixed_deposit,fd_rate_pct,fd_term_months,loan_balance,loan_type,loan_rate_pct,stocks,mutual_funds,...,employment_status,account_type,banking_channel,risk_tolerance,snapshot_date,data_origin,age_band,credit_band,tenure_band,churn_label,activity_label,card_label,mutual_funds_usd,loan_balance_usd,balance_usd,fixed_deposit_usd,stocks_usd,estimated_salary_usd,bitcoin_usd,balance_band
0,SYN004992,China,Female,82.0,38.0,593.0,CNY,0.140000,4.700457e+04,1.243979e+05,0.014547,60000.0,1057795.50,1.81,24.0,0.000000e+00,None,0.00,398589.50,6.822586e+04,...,Unemployed,Savings,Branch,High,2026-09-30,Synthetic classroom simulation,61+,580–669,9+,Retained,Active,No card,9551.620400,0.000000,6580.639800,148091.3700,55802.530000,17415.710200,872.8422,0–10k
1,SYN006753,Japan,Female,23.0,0.0,695.0,JPY,0.006700,3.682682e+06,5.648737e+06,0.129155,60000.0,0.00,0.00,0.0,1.020085e+07,Business,9.01,10887317.91,4.549358e+05,...,Employed,Savings,Online,High,2026-09-30,Synthetic classroom simulation,18–25,670–739,0–2,Retained,Active,No card,3048.069994,68345.689975,24673.970003,0.0000,72945.029997,37846.539977,7749.2712,10–50k
2,SYN003648,Denmark,Male,52.0,29.0,756.0,DKK,0.150000,6.029320e+04,9.740351e+05,0.000000,60000.0,0.00,0.00,0.0,0.000000e+00,None,0.00,1207645.27,0.000000e+00,...,Retired,Savings,Mobile,Medium,2026-09-30,Synthetic classroom simulation,46–60,740–799,9+,Retained,Inactive,Has card,0.000000,0.000000,9043.980000,0.0000,181146.790500,146105.269500,0.0000,0–10k
3,SYN007754,Indonesia,Female,82.0,18.0,665.0,IDR,0.000063,2.462060e+09,7.160867e+08,0.000000,60000.0,96832063.49,4.59,60.0,1.382741e+09,Home,8.27,0.00,8.646832e+08,...,Retired,Savings,Branch,Medium,2026-09-30,Synthetic classroom simulation,61+,580–669,9+,Churned,Inactive,No card,54475.040000,87112.680000,155109.800000,6100.4200,0.000000,45113.460000,0.0000,100k+
4,SYN004016,United States,Male,81.0,24.0,682.0,USD,1.000000,3.205794e+04,1.574851e+04,0.000000,60000.0,0.00,0.00,0.0,0.000000e+00,None,0.00,11690.41,0.000000e+00,...,Student,Premium,Online,Low,2026-09-30,Synthetic classroom simulation,61+,670–739,9+,Retained,Inactive,Has card,0.000000,0.000000,32057.940000,0.0000,11690.410000,15748.510000,0.0000,10–50k
5,SYN009459,United Kingdom,Male,25.0,3.0,568.0,GBP,1.250000,0.000000e+00,9.996970e+03,0.000000,60000.0,16106.82,7.60,24.0,0.000000e+00,None,0.00,3452.90,2.073354e+04,...,Unemployed,Current,Mobile,High,2026-09-30,Synthetic classroom simulation,18–25,300–579,3–5,Retained,Active,Has card,25916.925000,0.000000,0.000000,20133.5250,4316.125000,12496.212500,0.0000,Zero
6,SYN008883,Uzbekistan,Male,19.0,1.0,648.0,UZS,0.000080,1.043157e+09,1.314766e+09,0.067282,60000.0,0.00,0.00,0.0,7.176312e+07,Education,12.62,0.00,0.000000e+00,...,Self-Employed,Salary,Branch,Low,2026-09-30,Synthetic classroom simulation,18–25,580–669,0–2,Retained,Active,No card,0.000000,5741.050000,83452.570000,0.0000,0.000000,105181.290000,4036.9248,50–100k
7,SYN004699,Sweden,Female,41.0,4.0,709.0,SEK,0.095000,0.000000e+00,1.862792e+05,0.000000,60000.0,105602.00,6.20,12.0,0.000000e+00,None,0.00,0.00,0.000000e+00,...,Employed,Salary,Online,Low,2026-09-30,Synthetic classroom simulation,36–45,670–739,3–5,Retained,Active,Has card,0.000000,0.000000,0.000000,10032.1900,0.000000,17696.520200,0.0000,Zero
8,SYN005919,Ecuador,Male,64.0,11.0,650.0,USD,1.000000,1.192299e+04,4.536483e+04,0.205904,60000.0,25049.40,7.00,24.0,1.223262e+04,Personal,6.89,0.00,0.000000e+00,...,Retired,Premium,Online,Low,2026-09-30,Synthetic classroom simulation,61+,580–669,9+,Retained,Inactive,Has card,0.000000,12232.620000,11922.985000,25049.4000,0.000000,45364.830000,12354.2400,10–50k
9,SYN000264,Singapore,Female,66.0,29.0,814.0,SGD,0.740000,4.323150e+03,7.409569e+04,0.000000,60000.0,14586.47,1.44,60.0,7.143778e+04,Business,3.63,0.00,1.776166e+04,...,Student,Salary,Branch,High,2026-09-30,Synthetic classroom simulation,61+,800–85

Derived columns: ['age_band', 'credit_band', 'tenure_band', 'churn_label', 'activity_label', 'card_label', 'mutual_funds_usd', 'loan_balance_usd', 'balance_usd', 'fixed_deposit_usd', 'stocks_usd', 'estimated_salary_usd', 'bitcoin_usd', 'balance_band']


,column,q1,q3,iqr,lower,upper,flagged,method
0,age,35.000000,68.000000,33.000000,-14.500000,117.500000,0,IQR
1,tenure,4.000000,22.000000,18.000000,-23.000000,49.000000,0,IQR
2,credit_score,603.000000,715.000000,112.000000,435.000000,883.000000,32,IQR
3,fx_to_usd,0.007500,1.100000,1.092500,-1.631250,2.738750,144,IQR
4,bitcoin_btc,0.000000,0.048355,0.048355,-0.072533,0.120889,1591,IQR
5,bitcoin_price_usd,60000.000000,60000.000000,0.000000,60000.000000,60000.000000,0,IQR
6,fd_rate_pct,0.000000,4.140000,4.140000,-6.210000,10.350000,0,IQR
7,fd_term_months,0.000000,24.000000,24.000000,-36.000000,60.000000,0,IQR
8,loan_rate_pct,0.000000,10.720000,10.720000,-16.080000,26.800000,0,IQR
9,products_number,3.000000,5.000000,2.000000,0.000000,8.000000,0,IQR


,column,q1,q3,iqr,lower,upper,flagged,method
0,age,35.000000,68.000000,33.000000,-6.557737e+00,1.096318e+02,0,Z-score
1,tenure,4.000000,22.000000,18.000000,-1.992605e+01,4.767524e+01,0,Z-score
2,credit_score,603.000000,715.000000,112.000000,4.121613e+02,9.061316e+02,13,Z-score
3,fx_to_usd,0.007500,1.100000,1.092500,-1.541383e+00,2.500675e+00,412,Z-score
4,bitcoin_btc,0.000000,0.048355,0.048355,-9.328757e-01,1.118966e+00,144,Z-score
5,bitcoin_price_usd,60000.000000,60000.000000,0.000000,6.000000e+04,6.000000e+04,0,Z-score
6,fd_rate_pct,0.000000,4.140000,4.140000,-5.791836e+00,9.900912e+00,0,Z-score
7,fd_term_months,0.000000,24.000000,24.000000,-4.450647e+01,7.021219e+01,0,Z-score
8,loan_rate_pct,0.000000,10.720000,10.720000,-1.288360e+01,2.366414e+01,0,Z-score
9,products_number,3.000000,5.000000,2.000000,2.034075e-01,7.366797e+00,0,Z-score


Outliers are retained. IQR flags can include legitimate zeros or large holdings.


,column,count,mean,std,min,5%,25%,50%,75%,95%,max,median,variance_sample,skew,iqr,range,mode
0,age,9800.0,51.537041,1.936493e+01,18.000000,21.00000,35.000000,52.000000,68.000000,82.000000,8.500000e+01,52.000000,3.750004e+02,0.006093,33.000000,6.700000e+01,52.000
1,tenure,9800.0,13.874592,1.126688e+01,0.000000,0.00000,4.000000,11.000000,22.000000,36.000000,4.000000e+01,11.000000,1.269426e+02,0.630853,18.000000,4.000000e+01,0.000
2,credit_score,9800.0,659.146429,8.232838e+01,362.000000,521.00000,603.000000,660.000000,715.000000,798.000000,8.500000e+02,660.000000,6.777963e+03,-0.076780,112.000000,4.880000e+02,660.000
3,fx_to_usd,9800.0,0.479646,6.736763e-01,0.000040,0.00025,0.007500,0.140000,1.100000,1.410000,3.250000e+00,0.140000,4.538397e-01,1.963714,1.092500,3.249960e+00,1.100
4,bitcoin_btc,9627.0,0.093045,3.419737e-01,0.000000,0.00000,0.000000,0.000000,0.048355,0.455025,1.090208e+01,0.000000,1.169460e-01,11.460031,0.048355,1.090208e+01,0.000
5,bitcoin_price_usd,9800.0,60000.000000,0.000000e+00,60000.000000,60000.00000,60000.000000,60000.000000,60000.000000,60000.000000,6.000000e+04,60000.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,60000.000
6,fd_rate_pct,9800.0,2.054538,2.615458e+00,0.000000,0.00000,0.000000,0.000000,4.140000,7.230000,8.000000e+00,0.000000,6.840621e+00,0.867312,4.140000,8.000000e+00,0.000
7,fd_term_months,9800.0,12.852857,1.911978e+01,0.000000,0.00000,0.000000,0.000000,24.000000,60.000000,6.000000e+01,0.000000,3.655659e+02,1.443379,24.000000,6.000000e+01,0.000
8,loan_rate_pct,9800.0,5.390274,6.091290e+00,0.000000,0.00000,0.000000,3.410000,10.720000,16.390000,1.800000e+01,3.410000,3.710381e+01,0.611586,10.720000,1.800000e+01,0.000
9,has_bitcoin,9782.0,0.324576,4.682401e-01,0.000000,0.00000,0.000000,0.000000,1.000000,1.000000,1.000000e+00,0.000000,2.192487e-01,0.749445,1.000000,1.000000e+00,0.000


## 8. Segregation, dynamic sorting, filtering and extraction
Edit the controls dictionary and rerun this cell. The website provides these controls without editing code. Filters can combine with AND/OR; text search is literal, and sorting is stable. All summaries below use this same selected dataframe.

In [10]:
controls = {
    'categories': {'country':['India','United States','Germany']},
    'ranges': {'age':[25,65]},
    'logic':'AND',
    'search':'',
    'sort':[{'column':'balance_usd','ascending':False}]
}
selected=filter_frame(data,controls)
print(f'{len(selected):,} matching records out of {len(data):,}')
display(selected.head(20))
known=selected.churn.count()
print('Known outcomes:',known,'| churn rate:', f'{100*selected.churn.mean():.2f}%' if known else 'Unavailable')
country_summary=segments(selected,'country')
display(country_summary)
for group in ['gender','age_band','credit_band','tenure_band','industry','activity_label']:
    display(segments(selected,group))
display(pd.crosstab(selected.country,selected.churn_label,dropna=False))
high_balance=selected.loc[selected.balance_usd>=100000]
inactive_churned=selected.loc[(selected.active_member==0)&(selected.churn==1)]
print('Selected high-balance records:',len(high_balance),'| inactive and churned:',len(inactive_churned))

240 matching records out of 9,800

,customer_id,country,gender,age,tenure,credit_score,currency,fx_to_usd,balance,estimated_salary,bitcoin_btc,bitcoin_price_usd,fixed_deposit,fd_rate_pct,fd_term_months,loan_balance,loan_type,loan_rate_pct,stocks,mutual_funds,...,employment_status,account_type,banking_channel,risk_tolerance,snapshot_date,data_origin,age_band,credit_band,tenure_band,churn_label,activity_label,card_label,mutual_funds_usd,loan_balance_usd,balance_usd,fixed_deposit_usd,stocks_usd,estimated_salary_usd,bitcoin_usd,balance_band
7797,SYN003358,Germany,Male,28.0,9.0,654.0,EUR,1.100,353707.56,80517.44,0.000000,60000.0,10369.02,7.29,24.0,0.00,None,0.00,0.00,0.00,...,Self-Employed,Savings,Online,High,2026-09-30,Synthetic classroom simulation,26–35,580–669,9+,Retained,Active,Has card,0.000,0.00000,389078.31600,11405.92200,0.00000,88569.18400,0.0000,100k+
2848,SYN004618,India,Female,53.0,15.0,577.0,INR,0.012,17895065.00,1191201.67,0.000000,60000.0,0.00,0.00,0.0,1557850.83,Business,14.87,0.00,1880455.00,...,Self-Employed,Savings,Mixed,High,2026-09-30,Synthetic classroom simulation,46–60,300–579,9+,Retained,Active,Has card,22565.460,18694.20996,214740.78000,0.00000,0.00000,14294.42004,0.0000,100k+
6445,SYN005974,India,Male,29.0,7.0,643.0,INR,0.012,15641766.67,1791701.67,0.151432,60000.0,0.00,0.00,0.0,30894445.00,Personal,17.13,1917436.67,1376147.50,...,Retired,Salary,Mobile,Medium,2026-09-30,Synthetic classroom simulation,26–35,580–669,6–8,Retained,Inactive,Has card,16513.770,370733.34000,187701.20004,0.00000,23009.24004,21500.42004,9085.9038,100k+
3710,SYN000294,Germany,Male,41.0,0.0,699.0,EUR,1.100,162576.98,69907.95,0.000000,60000.0,0.00,0.00,0.0,0.00,None,0.00,24371.65,0.00,...,Retired,Savings,Branch,Medium,2026-09-30,Synthetic classroom simulation,36–45,670–739,0–2,Retained,Active,Has card,0.000,0.00000,178834.67800,0.00000,26808.81500,76898.74500,0.0000,100k+
185,SYN006829,Germany,Female,49.0,30.0,610.0,EUR,1.100,125550.46,56140.54,0.000000,60000.0,0.00,0.00,0.0,0.00,None,0.00,13546.70,11028.96,...,Employed,Savings,Mobile,Medium,2026-09-30,Synthetic classroom simulation,46–60,580–669,9+,Retained,Inactive,Has card,12131.856,0.00000,138105.50600,0.00000,14901.37000,61754.59400,0.0000,100k+
7935,SYN007071,India,Female,27.0,6.0,612.0,INR,0.012,11451700.00,7433395.83,0.000000,60000.0,1209111.67,4.85,60.0,8096737.50,Business,9.86,102746.67,1777495.00,...,Unemployed,Premium,Online,Low,2026-09-30,Synthetic classroom simulation,26–35,580–669,6–8,Retained,Active,Has card,21329.940,97160.85000,137420.40000,14509.34004,1232.96004,89200.74996,0.0000,100k+
6382,SYN002738,United States,Female,60.0,27.0,647.0,USD,1.000,137241.57,35657.37,0.487660,60000.0,15459.09,7.42,6.0,22664.17,Auto,6.16,63046.89,0.00,...,Student,Premium,Mixed,Medium,2026-09-30,Synthetic classroom simulation,46–60,580–669,9+,Retained,Inactive,Has card,0.000,22664.17000,137241.57000,15459.09000,63046.89000,35657.37000,29259.6090,100k+
2289,SYN008088,United States,Female,59.0,10.0,634.0,USD,1.000,134470.58,34138.70,0.000000,60000.0,22919.93,3.13,60.0,0.00,None,0.00,9880.03,6851.23,...,Unemployed,Current,Online,High,2026-09-30,Synthetic classroom simulation,46–60,580–669,9+,Retained,Active,Has card,6851.230,0.00000,134470.58000,22919.93000,9880.03000,34138.70000,0.0000,100k+
5236,SYN001604,Germany,Female,38.0,11.0,730.0,EUR,1.100,119268.48,40139.06,0.000000,60000.0,0.00,0.00,0.0,0.00,None,0.00,4633.10,0.00,...,Unemployed,Salary,Mixed,High,2026-09-30,Synthetic classroom simulation,36–45,670–739,9+,Retained,Active,No card,0.000,0.00000,131195.32800,0.00000,5096.41000,44152.96600,0.0000,100k+
6232,SYN002427,United States,Unknown,60.0,23.0,791.0,USD,1.000,124679.08,34698.76,0.006323,60000.0,0.00,0.00,0.0,4962.82,Education,6.34,0.00,33149.12,...,Unemployed,Savings,Online,Medium,2026-09-30,Synthetic classroom simulation,46–60,740–799,9+,Retained,Active,Has card,33149.120,4962.82000,124679.08000,0.00000,0.00000,34698.76000,379.3608,100k+


Known outcomes: 239 | churn rate: 16.32%


,country,customers,share_pct,known_outcomes,churned,retained,churn_rate_pct,mean_age,observed_age,mean_credit_score,observed_credit_score,mean_balance_usd,observed_balance_usd,mean_estimated_salary_usd,observed_estimated_salary_usd,mean_bitcoin_usd,observed_bitcoin_usd,mean_fixed_deposit_usd,observed_fixed_deposit_usd,mean_stocks_usd,observed_stocks_usd,mean_mutual_funds_usd,observed_mutual_funds_usd,mean_loan_balance_usd,observed_loan_balance_usd,small_sample
0,Germany,81,33.750000,81,16.0,65.0,19.753086,46.370370,81,652.802469,81,31266.721309,81,43875.244519,81,1868.940030,80,20283.347333,81,10183.012617,81,7670.805185,81,32222.174926,81,False
1,India,80,33.333333,80,11.0,69.0,13.750000,41.925000,80,660.512500,80,25295.630195,80,921375.403002,80,6547.477139,79,26864.471127,80,7755.679376,80,7751.516997,80,36884.427503,80,False
2,United States,79,32.916667,78,12.0,66.0,15.384615,46.037975,79,652.481013,79,24667.857215,79,43717.833038,79,677.043972,79,14138.809241,79,7225.359873,79,7298.736076,79,45910.976709,79,False


,gender,customers,share_pct,known_outcomes,churned,retained,churn_rate_pct,mean_age,observed_age,mean_credit_score,observed_credit_score,mean_balance_usd,observed_balance_usd,mean_estimated_salary_usd,observed_estimated_salary_usd,mean_bitcoin_usd,observed_bitcoin_usd,mean_fixed_deposit_usd,observed_fixed_deposit_usd,mean_stocks_usd,observed_stocks_usd,mean_mutual_funds_usd,observed_mutual_funds_usd,mean_loan_balance_usd,observed_loan_balance_usd,small_sample
1,Male,113,47.083333,112,19.0,93.0,16.964286,44.318584,113,654.654867,113,26409.753770,113,339063.266611,113,2921.434039,112,16795.518488,113,7099.115974,113,7077.739406,113,32825.074947,113,False
0,Female,110,45.833333,110,17.0,93.0,15.454545,45.918182,110,652.381818,110,27692.346943,110,378835.405593,110,3602.499055,109,23985.705382,110,10064.482236,110,8631.578662,110,32184.931648,110,False
2,Non-Binary,8,3.333333,8,2.0,6.0,25.000000,37.875000,8,674.250000,8,23754.825620,8,52991.714245,8,0.000000,8,32232.075630,8,4820.289750,8,3249.651500,8,146284.160375,8,True
4,Unknown,5,2.083333,5,0.0,5.0,0.000000,39.400000,5,707.600000,5,45967.803008,5,43164.361000,5,75.872160,5,9252.003792,5,4768.422808,5,8560.262400,5,11677.496000,5,True
3,Prefer Not To Say,4,1.666667,4,1.0,3.0,25.000000,47.000000,4,648.500000,4,13669.400490,4,22959.148750,4,0.000000,4,17159.237260,4,11096.054750,4,0.000000,4,177370.204000,4,True


...

,tenure_band,customers,share_pct,known_outcomes,churned,retained,churn_rate_pct,mean_age,observed_age,mean_credit_score,observed_credit_score,mean_balance_usd,observed_balance_usd,mean_estimated_salary_usd,observed_estimated_salary_usd,mean_bitcoin_usd,observed_bitcoin_usd,mean_fixed_deposit_usd,observed_fixed_deposit_usd,mean_stocks_usd,observed_stocks_usd,mean_mutual_funds_usd,observed_mutual_funds_usd,mean_loan_balance_usd,observed_loan_balance_usd,small_sample
3,9+,148,61.666667,148,22.0,126.0,14.864865,47.445946,148,662.878378,148,27136.885417,148,4.139066e+04,148,3188.259762,148,23068.584356,148,8098.586574,148,6157.892493,148,30138.299949,148,False
1,3–5,37,15.416667,37,8.0,29.0,21.621622,39.702703,37,640.405405,37,23417.934649,37,9.504836e+05,37,1262.419573,37,19818.863030,37,4888.844299,37,9573.150215,37,48780.972538,37,False
2,6–8,28,11.666667,27,1.0,26.0,3.703704,40.250000,28,647.392857,28,31740.513999,28,4.152172e+04,28,6053.734467,27,15296.243326,28,14051.109821,28,8411.409033,28,67145.941747,28,True
0,0–2,27,11.250000,27,8.0,19.0,29.629630,41.814815,27,642.074074,27,27168.835036,27,1.417086e+06,27,1470.368100,26,12345.612006,27,9006.395369,27,11739.356664,27,38602.473479,27,True


,industry,customers,share_pct,known_outcomes,churned,retained,churn_rate_pct,mean_age,observed_age,mean_credit_score,observed_credit_score,mean_balance_usd,observed_balance_usd,mean_estimated_salary_usd,observed_estimated_salary_usd,mean_bitcoin_usd,observed_bitcoin_usd,mean_fixed_deposit_usd,observed_fixed_deposit_usd,mean_stocks_usd,observed_stocks_usd,mean_mutual_funds_usd,observed_mutual_funds_usd,mean_loan_balance_usd,observed_loan_balance_usd,small_sample
10,Technology,29,12.083333,29,6.0,23.0,20.689655,47.517241,29,664.172414,29,28003.940480,29,5.265295e+04,29,3987.857483,29,22572.886897,29,11777.389034,29,4465.693828,29,32971.282137,29,True
11,Transport,25,10.416667,25,2.0,23.0,8.000000,42.880000,25,634.480000,25,23404.351232,25,4.541239e+04,25,7079.111568,25,36524.279482,25,13332.190398,25,10602.941960,25,29674.177560,25,True
5,Hospitality,23,9.583333,23,2.0,21.0,8.695652,45.173913,23,654.173913,23,29885.471432,23,3.304036e+04,23,2861.190927,22,12416.734127,23,7170.519694,23,6835.734696,23,51277.421212,23,True
8,Public Services,21,8.750000,21,2.0,19.0,9.523810,43.523810,21,655.761905,21,30590.378147,21,1.801855e+06,21,3756.379714,21,24901.599760,21,11649.655046,21,6010.095189,21,25334.036764,21,True
3,Finance,21,8.750000,20,5.0,15.0,25.000000,45.761905,21,657.666667,21,31067.626288,21,5.159938e+04,21,909.857286,21,10765.499050,21,3279.022383,21,9092.949095,21,23335.229099,21,True
7,Other,19,7.916667,19,2.0,17.0,10.526316,42.894737,19,659.105263,19,43171.669678,19,5.348933e+04,19,5079.978284,19,10658.540947,19,4712.979206,19,7137.962737,19,62988.631154,19,True
6,Manufacturing,19,7.916667,19,3.0,16.0,15.789474,43.263158,19,655.263158,19,23438.502522,19,4.348928e+04,19,1050.668589,19,14033.378526,19,3003.039688,19,6791.553208,19,54540.694472,19,True
4,Healthcare,19,7.916667,19,3.0,16.0,15.789474,43.368421,19,660.210526,19,34568.306632,19,1.796265e+06,19,821.383267,18,19034.669482,19,7279.357368,19,10435.532415,19,10444.926320,19,True
0,Agriculture,19,7.916667,19,4.0,15.0,21.052632,45.578947,19,667.684211,19,26028.499682,19,4.389777e+04,19,4697.885242,19,32189.119533,19,4581.775737,19,8207.604373,19,65904.738476,19,True
2,Education,15,6.250000,15,5.0,10.0,33.333333,47.866667,15,659.266667,15,8491.244400,15,3.112197e+04,15,761.989200,15,29829.776669,15,6832.274205,15,12821.202995,15,25266.342003,15,True


,activity_label,customers,share_pct,known_outcomes,churned,retained,churn_rate_pct,mean_age,observed_age,mean_credit_score,observed_credit_score,mean_balance_usd,observed_balance_usd,mean_estimated_salary_usd,observed_estimated_salary_usd,mean_bitcoin_usd,observed_bitcoin_usd,mean_fixed_deposit_usd,observed_fixed_deposit_usd,mean_stocks_usd,observed_stocks_usd,mean_mutual_funds_usd,observed_mutual_funds_usd,mean_loan_balance_usd,observed_loan_balance_usd,small_sample
0,Active,153,63.75,152,19.0,133.0,12.500000,44.359477,153,651.274510,153,26980.278036,153,504422.118774,153,3479.090353,152,18165.69210,153,6687.360528,153,8037.257411,153,33398.008295,153,False
1,Inactive,87,36.25,87,20.0,67.0,22.988506,45.517241,87,662.287356,87,27322.218644,87,40701.743491,87,2225.937635,86,24479.58463,87,11412.823463,87,6762.716620,87,46871.520230,87,False


churn_label,Churned,Retained,Unknown
country,,,
Germany,16,65,0
India,11,69,0
United States,12,66,1


Selected high-balance records: 17 | inactive and churned: 20


## 9. Interactive visualization gallery
All charts in this section use `selected`, not the full source. Hover, zoom and export with the Plotly toolbar. Sample sizes and missingness matter when comparing countries. Holding more of an asset does not establish that the asset performs better.

In [11]:
if not selected.empty:
    px.bar(country_summary,x='country',y='customers',title='Selected records by country').show()
    px.pie(selected,names='churn_label',hole=.6,title='Selected churn outcomes; Unknown shown separately').show()
    px.histogram(selected,x='age',color='churn_label',title='Selected age distribution').show()
    px.scatter(selected,x='age',y='balance_usd',color='churn_label',opacity=.45,title='Age and account balance (USD)').show()
    px.box(selected,x='country',y='balance_usd',color='country',title='Balance box plots (USD)').show()
    px.violin(selected,x='country',y='balance_usd',color='country',box=True,points=False,title='Balance violin distributions (USD)').show()
    nums=selected.select_dtypes('number').drop(columns=list(MONEY)+['customer_id'],errors='ignore')
    px.imshow(nums.corr(),zmin=-1,zmax=1,color_continuous_scale='RdBu_r',title='Pairwise-complete correlation matrix').show()
    tenure=selected.groupby('tenure',as_index=False).balance_usd.mean()
    px.line(tenure,x='tenure',y='balance_usd',markers=True,title='Balance by tenure — cross-sectional, not historical').show()
    assets=['balance_usd','bitcoin_usd','fixed_deposit_usd','stocks_usd','mutual_funds_usd']
    asset_summary=selected.groupby('country')[assets].sum(min_count=1).reset_index().melt(id_vars='country',var_name='asset',value_name='usd')
    px.bar(asset_summary,x='country',y='usd',color='asset',title='Recorded asset allocation by selected country (USD)').show()
else:
    print('No matching records. Broaden controls and rerun section 8.')

## 10. Export the evidence and reproduce the website
Exports include cleaned and transformed data, the exact filtered extraction, before/after profile, column-level decisions, cell-level provenance and an Excel report. CSV text is escaped to avoid spreadsheet formula execution.

To open the full website, run `start_dashboard.bat` from the project folder, then visit http://127.0.0.1:8765. The website uses this same pipeline with cleaning preview/apply, undo/redo, uploads, 23 chart choices, sorting, extraction and report exports.

In [12]:
excel_safe(cleaned).to_csv(OUT/'banking_cleaned.csv',index=False)
excel_safe(data).to_csv(OUT/'banking_transformed.csv',index=False)
excel_safe(selected).to_csv(OUT/'filtered_extraction.csv',index=False)
excel_safe(cell_audit).to_csv(OUT/'cell_level_audit.csv',index=False)
decisions.to_csv(OUT/'cleaning_decisions.csv',index=False)
comparison.to_csv(OUT/'quality_before_after.csv',index=False)
pd.DataFrame(audit).to_csv(OUT/'cleaning_audit.csv',index=False)
(OUT/'analytics_report.xlsx').write_bytes(workbook(selected,audit,after))
manifest={'input_sha256':hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),'input_rows':len(raw),'output_rows':len(cleaned),
          'filters':controls,'plan':plan,'synthetic_source':True,'pandas':pd.__version__,'numpy':np.__version__}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved:',*[p.name for p in OUT.iterdir() if p.is_file()],sep='\n')

Saved:
analytics_report.xlsx
banking_cleaned.csv
banking_transformed.csv
cell_level_audit.csv
cleaning_audit.csv
cleaning_decisions.csv
filtered_extraction.csv
quality_before_after.csv
run_manifest.json


## 11. Isolated missing-value demonstration
This is an explicitly artificial four-row illustration, separate from the supplied dataset. It demonstrates empty text, numeric placeholders, the preservation of zero, and currency-aware medians. It is not presented as evidence about the banking source.

In [13]:
demo=pd.DataFrame({'customer_id':['demo1','demo2','demo3','demo4'], 'currency':['USD','USD','JPY','JPY'],
                   'balance':['0','NA','10000',' '], 'gender':['Female','  ','Male','null'], 'churn':['1','0','','1']})
demo_clean,demo_log=clean(demo,{'balance':'group_median','gender':'unknown'})
display(demo);display(demo_clean);display(pd.DataFrame(demo_log))
assert demo_clean.balance.tolist()==[0,0,10000,10000]
assert demo_clean.churn.isna().sum()==1
assert demo.loc[1,'balance']=='NA'
pd.testing.assert_frame_equal(raw,raw_snapshot)
assert segments(data,'country').customers.sum()==len(data)
assert len(selected)==len(filter_frame(data,controls))
print('Reproducibility and integrity checks passed.')

,customer_id,currency,balance,gender,churn
0,demo1,USD,0,Female,1
1,demo2,USD,NA,,0
2,demo3,JPY,10000,Male,
3,demo4,JPY,,null,1


,customer_id,currency,balance,gender,churn
0,demo1,USD,0.0,Female,1.0
1,demo2,USD,0.0,Unknown,0.0
2,demo3,JPY,10000.0,Male,NaN
3,demo4,JPY,10000.0,Unknown,1.0


,timestamp,column,method,affected,reason,replacement
0,2026-10-12T04:22:34.573803+00:00,balance,normalize / validate,2,Text/null missing: 2; invalid: 0. Invalid valu...,NaN
1,2026-10-12T04:22:34.573812+00:00,gender,normalize / validate,2,Text/null missing: 2; invalid: 0. Invalid valu...,NaN
2,2026-10-12T04:22:34.573814+00:00,churn,normalize / validate,1,Text/null missing: 1; invalid: 0. Invalid valu...,NaN
3,2026-10-12T04:22:34.587805+00:00,all,exact duplicate removal,0,Identical source rows only; conflicting IDs re...,NaN
4,2026-10-12T04:22:34.587805+00:00,balance,group_median,2,Money is skewed and currencies differ: use the...,within-currency median
5,2026-10-12T04:22:34.587805+00:00,gender,unknown,2,Use Unknown to preserve missingness; mode is a...,Unknown


Reproducibility and integrity checks passed.


## 12. Conclusions and viva discussion
1. Report observed source defects and unresolved cells; do not claim all data is now true or complete.
2. Explain why numeric distributions, valid zeros and currency units determine cleaning choices.
3. Disclose the bias introduced by median/mode imputation; compare before/after profiles.
4. Explain repeated customer IDs separately from exact row duplicates. Dashboard counts are records, not necessarily unique people.
5. Compare asset holdings and ownership only within the selected records, with observed counts. These are not real-world investment recommendations.
6. The source is a synthetic snapshot; it cannot support real-time market or historical trend claims.
7. For production: add authenticated multi-user isolation, encrypted storage, retention controls, independent schema review, background processing and deployment monitoring. The delivered website is designed for a local academic demonstration.